# 03 — Concept drift (when P(Y|X) changes)

On **day 70** the synthetic finance policy tightens: categories that auto-cleared now require review. Features look similar; **the correct label** changes. That is **concept drift**.

Watch **calibration** (ECE) and reviewer queue volume—not accuracy alone. In this generator, accuracy can move in a misleading direction while probabilities stop meaning what they did under the old policy.


In [ ]:
# From repo root: pip install -e ".[dev]"
%matplotlib inline

import pandas as pd

from drift_lab import StreamConfig, build_ledger_route_model, generate_stream
from drift_lab.analysis import attach_predictions, outcome_summary
from drift_lab.metrics import rolling_accuracy
from drift_lab.viz import calibration_figure, rolling_accuracy_figure

cfg = StreamConfig()
model = build_ledger_route_model(cfg)
df = generate_stream("concept_abrupt", cfg)
scored = attach_predictions(model, df)

shift = cfg.concept_shift_day
pre = scored["day"] < shift
post = scored["day"] >= shift
print("Concept shift day:", shift)


## Accuracy and ECE pre/post shock


In [ ]:
summary = pd.DataFrame(
    {
        "window": ["pre-shift", "post-shift"],
        **{
            k: [
                outcome_summary(
                    scored.loc[pre, "label"],
                    scored.loc[pre, "pred_label"],
                    scored.loc[pre, "prob_review"],
                )[k],
                outcome_summary(
                    scored.loc[~pre, "label"],
                    scored.loc[~pre, "pred_label"],
                    scored.loc[~pre, "prob_review"],
                )[k],
            ]
            for k in ["accuracy", "ece", "positive_rate", "mean_score"]
        },
    }
)
summary.round(4)


## Reliability diagrams


In [ ]:
fig_pre, _ = calibration_figure(scored.loc[pre, "label"], scored.loc[pre, "prob_review"])
fig_pre.suptitle("Pre-shift calibration", y=1.02)
fig_post, _ = calibration_figure(scored.loc[~pre, "label"], scored.loc[~pre, "prob_review"])
fig_post.suptitle("Post-shift calibration", y=1.02)


## Rolling accuracy with frozen weights


In [ ]:
acc_df = rolling_accuracy(scored["label"].values, scored["pred_label"].values, window=800)
fig, roll_meta = rolling_accuracy_figure(
    acc_df,
    title="Rolling accuracy after label semantics change (weights frozen)",
)
roll_meta


## Takeaway

1. Version **`label_policy`** with every label row—treat policy releases like schema migrations.
2. Page on **calibration / queue depth** when accuracy still looks acceptable.
3. Retrain on **post-shock labels only**, or weight by policy version; do not mix incompatible outcomes in one loss.
